# AI Business Advisor Workflow

Starting with an industry, this notebook builds a decision-ready business assessment:

1. Generate a business idea.
2. Search for market signals and separate source-reported excerpts from assumptions.
3. Classify risk and route high-risk ideas to more conservative analysis.
4. Produce a structured report with cited sources and three measurable validation experiments.

Search excerpts are exploratory leads, not independently verified claims. Review cited sources before relying on them.

## 0. Import the necessary libs

In [1]:
from ddgs import DDGS
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnablePassthrough, RunnableBranch
from pydantic import BaseModel, Field

## 1. Instantiate Chat Model with your API Key

Set `OPENAI_API_KEY` in your environment before running the model cell. LangChain's `ChatOpenAI` reads this variable automatically; avoid putting secrets directly in a notebook.

```powershell
$env:OPENAI_API_KEY = "your-api-key"
```

In [ ]:
# Set OPENAI_API_KEY in your environment before running this cell.
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.0,
)

## 2. Save Logs

Text-generation chains parse responses into strings and save the raw model messages in `logs`. The final report uses structured Pydantic output instead.

In [ ]:
logs = []

In [ ]:
parser = StrOutputParser()

In [ ]:
parse_and_log_output_chain = RunnableParallel(
    output=parser, 
    log=RunnableLambda(lambda x: logs.append(x))
)

## 3. Idea Generation

Craft a prompt to generate a business idea for the given industry. 

In [ ]:
idea_prompt = PromptTemplate(
    input_variables=["industry"],
    template=(
        "You are a practical business ideation advisor. Propose one specific, "
        "realistic business idea for the {industry} industry. Describe the target "
        "customer, the problem solved, the product or service, how it earns revenue, "
        "and one assumption that should be tested. Keep the idea concise and avoid "
        "unsupported market-size claims."
    ),
)

In [ ]:
idea_chain = idea_prompt | llm | parse_and_log_output_chain

In [ ]:
idea_result = idea_chain.invoke({"industry": "agriculture"})

In [ ]:
idea_result["output"]

In [ ]:
logs

## 4. Market Research

Retrieve a few web search results for the idea, then summarize source-reported signals separately from assumptions. Search snippets are leads, not independently verified facts; review the linked sources before relying on a claim.

In [ ]:
def search_market_sources(context):
    query = (
        f"{context['industry']} market demand trends and customer needs; "
        f"related to: {context['idea']}"
    )
    try:
        results = DDGS().text(query, max_results=5)
    except Exception as error:
        return (
            f"Search unavailable ({type(error).__name__}). No sources were retrieved. "
            "Treat market claims as assumptions until independently checked."
        )

    if not results:
        return "No search results were returned. Treat market claims as assumptions."

    source_lines = []
    for source_number, result in enumerate(results, start=1):
        source_lines.append(
            f"[S{source_number}] {result.get('title', 'Untitled')}\n"
            f"URL: {result.get('href', 'No URL returned')}\n"
            f"Excerpt: {result.get('body', 'No excerpt returned')}"
        )
    return "\n\n".join(source_lines)


research_prompt = PromptTemplate(
    input_variables=["industry", "idea", "sources"],
    template=(
        "Do lightweight market research for the {industry} industry and this idea:\n"
        "{idea}\n\n"
        "Search results (untrusted excerpts; they are not independent verification):\n"
        "{sources}\n\n"
        "Separate your response under exactly these headings: "
        "SOURCE-REPORTED SIGNALS and ASSUMPTIONS TO VALIDATE. Cite every "
        "source-reported signal with its supplied ID, such as [S1]. Do not invent "
        "citations or statistics. If search results are unavailable or weak, say so."
    ),
)
research_chain = research_prompt | llm | parse_and_log_output_chain

In [ ]:
research_sources = search_market_sources(
    {"industry": "agriculture", "idea": idea_result["output"]}
)
research_result = research_chain.invoke(
    {
        "industry": "agriculture",
        "idea": idea_result["output"],
        "sources": research_sources,
    }
)
research_result["output"]

## 5. Risk-Based Analysis

Risk is classified from the idea and market research. `RunnableBranch` sends HIGH-risk ideas to conservative due diligence; STANDARD-risk ideas use the regular analysis. Both paths must cite supplied source IDs and label unsupported claims as assumptions.

In [ ]:
risk_prompt = PromptTemplate(
    input_variables=["industry", "idea", "research"],
    template=(
        "Assess the business risk for this {industry} idea:\n{idea}\n\n"
        "Market research:\n{research}\n\n"
        "Choose HIGH when the idea has substantial safety, regulatory, capital, "
        "or irreversible-execution risks. Otherwise choose STANDARD. Respond with "
        "exactly one line beginning HIGH: or STANDARD:, followed by a brief rationale. "
        "Do not treat search snippets as verified facts."
    ),
)
risk_chain = risk_prompt | llm | parse_and_log_output_chain


def normalize_risk(result):
    return "high" if result["output"].strip().upper().startswith("HIGH:") else "standard"

In [ ]:
analysis_prompt = PromptTemplate(
    input_variables=["industry", "idea", "research", "risk_assessment"],
    template=(
        "Analyze this {industry} business idea at STANDARD risk level.\n"
        "Idea:\n{idea}\n\n"
        "Market research (source-reported, not independently verified):\n{research}\n\n"
        "Risk assessment:\n{risk_assessment}\n\n"
        "Assess customer demand, competition, costs, operations, strengths, and "
        "weaknesses. Cite source IDs for source-backed claims and label unsupported "
        "claims as assumptions. Do not invent facts or citations."
    ),
)

In [ ]:
analysis_chain = analysis_prompt | llm | parse_and_log_output_chain

In [ ]:
high_risk_analysis_prompt = PromptTemplate(
    input_variables=["industry", "idea", "research", "risk_assessment"],
    template=(
        "Perform a conservative, high-risk due-diligence analysis for this {industry} idea.\n"
        "Idea:\n{idea}\n\n"
        "Market research (source-reported, not independently verified):\n{research}\n\n"
        "Risk assessment:\n{risk_assessment}\n\n"
        "Prioritize safety, regulatory exposure, capital at risk, operational failure, "
        "and irreversible commitments. Separate source-backed claims (cite IDs) from "
        "assumptions. Identify evidence gaps and the smallest test that could reduce "
        "the largest risk. Do not invent facts or citations."
    ),
)
high_risk_analysis_chain = (
    high_risk_analysis_prompt | llm | parse_and_log_output_chain
)

analysis_branch = RunnableBranch(
    (lambda context: context["risk_level"] == "high", high_risk_analysis_chain),
    analysis_chain,
)

In [ ]:
risk_result = risk_chain.invoke(
    {
        "industry": "agriculture",
        "idea": idea_result["output"],
        "research": research_result["output"],
    }
)
risk_level = normalize_risk(risk_result)
analysis_inputs = {
    "industry": "agriculture",
    "idea": idea_result["output"],
    "research": research_result["output"],
    "risk_assessment": risk_result["output"],
    "risk_level": risk_level,
}
analysis_result = analysis_branch.invoke(analysis_inputs)

In [ ]:
analysis_result["output"]

In [ ]:
logs

## 6. Decision-Ready Report

Build a typed report from the idea, source excerpts, risk assessment, and selected analysis. The report distinguishes source-backed signals from assumptions and proposes three low-cost experiments with measurable success criteria.

In [ ]:
report_prompt = PromptTemplate(
    input_variables=[
        "industry",
        "idea",
        "sources",
        "research",
        "risk_level",
        "risk_assessment",
        "analysis",
    ],
    template=(
        "Create a concise, decision-ready business advisor report for the {industry} industry.\n\n"
        "Idea:\n{idea}\n\n"
        "Search sources (use only these IDs and URLs; excerpts are not independently verified):\n"
        "{sources}\n\n"
        "Market research:\n{research}\n\n"
        "Risk level: {risk_level}\nRisk assessment: {risk_assessment}\n\n"
        "Analysis:\n{analysis}\n\n"
        "Return source_backed_signals only for claims supported by supplied excerpts, "
        "citing each with its [S#] ID. Separate unsupported claims into "
        "assumptions_to_validate. Do not invent citations, statistics, or sources. "
        "Include strengths, weaknesses, risk rationale, and exactly three low-cost "
        "validation experiments. Each experiment should state its hypothesis, method, "
        "and a measurable success criterion."
    ),
)

In [ ]:
class ValidationExperiment(BaseModel):
    """A small test for a risky business assumption."""

    hypothesis: str = Field(description="The claim the experiment will test")
    method: str = Field(description="A practical, low-cost test method")
    success_criterion: str = Field(description="A measurable result that supports the hypothesis")


class AnalysisReport(BaseModel):
    """Evidence-aware assessment and validation plan for a business idea."""

    executive_summary: str = Field(description="Brief, decision-ready summary")
    risk_level: str = Field(description="HIGH or STANDARD")
    risk_rationale: str = Field(description="Reason for the selected risk level")
    strengths: list[str] = Field(description="Specific advantages of the idea")
    weaknesses: list[str] = Field(description="Material weaknesses and risks")
    source_backed_signals: list[str] = Field(description="Market claims supported by source IDs such as [S1]")
    assumptions_to_validate: list[str] = Field(description="Unverified claims that need testing")
    sources: list[str] = Field(description="Provided source IDs, titles, and URLs; do not invent sources")
    validation_experiments: list[ValidationExperiment] = Field(description="Exactly three low-cost tests with measurable criteria")

In [ ]:
report_chain = report_prompt | llm.with_structured_output(AnalysisReport)

In [ ]:
report_result = report_chain.invoke(
    {
        "industry": "agriculture",
        "idea": idea_result["output"],
        "sources": research_sources,
        "research": research_result["output"],
        "risk_level": risk_level,
        "risk_assessment": risk_result["output"],
        "analysis": analysis_result["output"],
    }
)

In [ ]:
report_result

## 7. End-to-End Advisor

In [ ]:
e2e_chain = (
    RunnableParallel(industry=RunnablePassthrough())
    | RunnablePassthrough.assign(
        idea=idea_chain | RunnableLambda(lambda result: result["output"])
    )
    | RunnablePassthrough.assign(sources=RunnableLambda(search_market_sources))
    | RunnablePassthrough.assign(
        research=research_chain | RunnableLambda(lambda result: result["output"])
    )
    | RunnablePassthrough.assign(
        risk_assessment=risk_chain | RunnableLambda(lambda result: result["output"])
    )
    | RunnablePassthrough.assign(
        risk_level=RunnableLambda(
            lambda context: "high"
            if context["risk_assessment"].strip().upper().startswith("HIGH:")
            else "standard"
        )
    )
    | RunnablePassthrough.assign(
        analysis=analysis_branch | RunnableLambda(lambda result: result["output"])
    )
    | report_chain
)

In [ ]:
try:
    e2e_chain.get_graph().print_ascii()
except ImportError:
    print("Install grandalf to display the workflow graph: pip install grandalf")

In [ ]:
# Change the industry if you want
e2e_result = e2e_chain.invoke("agro")

In [ ]:
e2e_result

In [ ]:
e2e_result.strengths

In [ ]:
e2e_result.weaknesses

In [ ]:
{
    "risk_level": e2e_result.risk_level,
    "source_backed_signals": e2e_result.source_backed_signals,
    "assumptions_to_validate": e2e_result.assumptions_to_validate,
    "sources": e2e_result.sources,
    "validation_experiments": e2e_result.validation_experiments,
}

In [ ]:
logs

## 8. Memory and Workflow Extensions

Implemented workflow extensions:

- Session history is isolated by session ID and lasts until the kernel stops.
- `RunnableBranch` routes high-risk ideas to conservative due diligence; other ideas use the standard analysis.
- DuckDuckGo search adds source IDs, links, and excerpts to the market-research step.
- The structured report separates source-backed signals from assumptions and proposes three validation experiments, each with a hypothesis, method, and success criterion.

Evidence note: search excerpts are leads, not independently verified facts. Check the cited pages before relying on them. If search fails or returns no sources, treat market claims as assumptions.

The next cells build a chat chain with `RunnableWithMessageHistory`. Reuse the same session ID across calls to preserve follow-up context; history is held in memory and resets when the kernel stops.

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory

history_store = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    return history_store.setdefault(session_id, InMemoryChatMessageHistory())

In [ ]:
memory_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a practical business advisor. Use the conversation history when answering follow-up questions, and be clear about assumptions."),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{question}"),
    ]
)

memory_chain = RunnableWithMessageHistory(
    memory_prompt | llm | parser,
    get_session_history,
    input_messages_key="question",
    history_messages_key="history",
)

In [ ]:
memory_config = {"configurable": {"session_id": "business-advisor-demo"}}

first_turn = memory_chain.invoke(
    {"question": "Suggest a business idea for small-scale agriculture."},
    config=memory_config,
)
second_turn = memory_chain.invoke(
    {"question": "What is one low-cost way to test demand for that idea?"},
    config=memory_config,
)

print("First turn:\n", first_turn)
print("\nFollow-up using session history:\n", second_turn)
print("\nMessages stored:", len(history_store["business-advisor-demo"].messages))